# Books & Interactions Processing Pipeline: Mystery, Thriller & Crime

Final curation pipeline for the Goodreads Mystery, Thriller & Crime category. The cleaning and feature engineering choices are grounded in `EDA_Mystery.ipynb`: role-filtered authorship, mystery shelf taxonomy, separated demand vs satisfaction signals, cold-start routing, user rating bias, valid publication years, and transformed long-tail counts.

## Section 0 — Setup

Use the shared category configuration so raw and processed paths stay aligned with the rest of the project. Both books and interactions are processed in chunks to avoid holding large Goodreads files in memory.

In [1]:
from __future__ import annotations

from collections import Counter
from datetime import datetime, timezone
import gc
import json
import os
from pathlib import Path
import sys
from typing import Any

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import BOOK_NUMERIC_COLUMNS, CATEGORIES, GOODREADS_DATE_COLUMNS
from src.utils.cleaning import empty_strings_to_na, normalize_review_text, parse_bool_series
from src.utils.io import read_parquet_chunks, safe_write_parquet

CATEGORY = 'mystery_thriller_crime'
cfg = CATEGORIES[CATEGORY]
OUTPUT_DIR = cfg.processed_dir
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BOOKS_CURATED_PATH = OUTPUT_DIR / 'books_curated.parquet'
INTERACTIONS_CURATED_PATH = OUTPUT_DIR / 'interactions_curated.parquet'
USER_FEATURES_PATH = OUTPUT_DIR / 'user_features.parquet'
BOOK_INTERACTION_FEATURES_PATH = OUTPUT_DIR / 'book_interaction_features.parquet'
SUMMARY_PATH = OUTPUT_DIR / 'curation_summary.json'
BOOKS_IN_PATH = cfg.interim_dir / 'books_reduced.parquet'
INTERACTIONS_IN_PATH = cfg.interim_dir / 'interactions_reduced.parquet'

BOOK_CHUNKSIZE = 50_000
INTERACTION_CHUNKSIZE = 200_000
COLD_START_THRESHOLD = 10
PUBLICATION_YEAR_MIN = 1450
PUBLICATION_YEAR_MAX = 2026
THEME_COUNT_COL = 'genre_theme_count'
VALID_ENGAGEMENT_MODES = {'shelf_only', 'rating_only', 'review', 'read_no_rating'}
PUBLICATION_PERIOD_BINS = [1450, 1800, 1900, 1950, 1980, 2000, 2010, 2027]
PUBLICATION_PERIOD_LABELS = ['1450-1799', '1800-1899', '1900-1949', '1950-1979', '1980-1999', '2000-2009', '2010-2026']

print(cfg.display_name)
print(BOOKS_IN_PATH)
print(INTERACTIONS_IN_PATH)
print(OUTPUT_DIR)

Mystery, Thriller & Crime
/home/nakato/projects/BigBook/data/interim/mystery_thriller_crime/books_reduced.parquet
/home/nakato/projects/BigBook/data/interim/mystery_thriller_crime/interactions_reduced.parquet
/home/nakato/projects/BigBook/data/processed/mystery_thriller_crime


## Section 1 — Helper Functions

Helpers are local to this notebook to keep `src/utils` unchanged. They mirror the EDA decisions and keep the full pipeline chunk-safe.

In [2]:
THEME_GROUPS = {'crime_detective': ['crime', 'detective', 'mystery', 'whodunit'], 'thriller': ['thriller', 'suspense'], 'cozy_mystery': ['cozy-mystery', 'cozy', 'cosy-mystery'], 'police_procedural': ['police-procedural', 'procedural'], 'psychological': ['psychological-thriller', 'psychological'], 'historical_mystery': ['historical-mystery', 'historical'], 'noir_hardboiled': ['noir', 'hard-boiled', 'hardboiled']}


def parse_goodreads_dates_inplace(df: pd.DataFrame) -> pd.DataFrame:
    for col in GOODREADS_DATE_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
    return df


def safe_to_numeric_inplace(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def parse_bool_column(series: pd.Series) -> pd.Series:
    return parse_bool_series(series) if len(series) else pd.Series(dtype='boolean')


def json_dumps_or_na(value: Any) -> str | pd.NA:
    if value is None:
        return pd.NA
    if not isinstance(value, (list, dict)) and pd.isna(value):
        return pd.NA
    return json.dumps(value, ensure_ascii=False)


def clean_string_column(series: pd.Series) -> pd.Series:
    return series.astype('string').str.strip().replace('', pd.NA)


def normalize_review_text_series(series: pd.Series) -> pd.Series:
    out = pd.Series(pd.NA, index=series.index, dtype='object')
    mask = series.notna()
    if mask.any():
        out.loc[mask] = series.loc[mask].map(normalize_review_text)
    return out


def features_from_nested(series: pd.Series, extractor) -> pd.DataFrame:
    return pd.DataFrame(series.map(extractor).tolist(), index=series.index)


def has_theme(names: list[str], keywords: list[str]) -> bool:
    return any(any(keyword in name for keyword in keywords) for name in names)


def as_list(value: Any) -> list:
    if hasattr(value, "tolist"):
        value = value.tolist()
    return value if isinstance(value, list) else []

def extract_author_features(authors: Any) -> dict[str, Any]:
    authors = as_list(authors)
    valid = [item for item in authors if isinstance(item, dict) and item.get('author_id')]
    role_filtered = [item for item in valid if str(item.get('role', '')) == '']
    fallback = valid[0] if valid else None
    primary = role_filtered[0] if role_filtered else None
    return {
        'primary_author_id_role_filtered': str(primary.get('author_id')) if primary else pd.NA,
        'author_fallback_id': str(fallback.get('author_id')) if fallback else pd.NA,
        'author_count': len(valid),
        'non_primary_role_count': sum(1 for item in valid if str(item.get('role', '')) != ''),
        'primary_author_role': str(primary.get('role', '')) if primary else pd.NA,
    }


def extract_series_features(series_value: Any) -> dict[str, Any]:
    series_value = as_list(series_value)
    count = len(series_value)
    return {
        'series_count': count,
        'is_in_series': count > 0,
        'series_json': json_dumps_or_na(series_value) if count else pd.NA,
    }


def extract_similar_books_features(value: Any) -> dict[str, Any]:
    value = as_list(value)
    count = len(value)
    return {
        'similar_books_count': count,
        'similar_books_json': json_dumps_or_na(value) if count else pd.NA,
    }


def extract_shelf_features(shelves: Any, top_n: int = 30) -> dict[str, Any]:
    shelves = as_list(shelves)
    if not shelves:
        base = {'to_read_count': np.nan, 'shelf_count': 0, 'top_shelves': pd.NA, 'top_shelves_json': pd.NA}
        for theme in THEME_GROUPS:
            base[f'theme_{theme}'] = False
        base[THEME_COUNT_COL] = 0
        return base

    cleaned = []
    to_read_count = np.nan
    for item in shelves:
        if not isinstance(item, dict):
            continue
        name = item.get('name')
        if not name:
            continue
        count = pd.to_numeric(item.get('count'), errors='coerce')
        name = str(name)
        if name.lower() == 'to-read':
            to_read_count = count
        cleaned.append({'name': name, 'count': None if pd.isna(count) else int(count)})

    top = cleaned[:top_n]
    names = [item['name'].lower() for item in cleaned]
    base = {
        'to_read_count': to_read_count,
        'shelf_count': len(cleaned),
        'top_shelves': '|'.join(item['name'] for item in top) if top else pd.NA,
        'top_shelves_json': json.dumps(top, ensure_ascii=False) if top else pd.NA,
    }
    theme_count = 0
    for theme, keywords in THEME_GROUPS.items():
        value = has_theme(names, keywords)
        base[f'theme_{theme}'] = value
        theme_count += int(value)
    base[THEME_COUNT_COL] = theme_count
    return base


def add_log_and_cap_features(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for col in columns:
        if col not in df.columns:
            continue
        values = pd.to_numeric(df[col], errors='coerce')
        df[f'{col}_log1p'] = np.log1p(values.clip(lower=0))
        cap = float(values.quantile(0.99))
        df[f'{col}_p99_capped'] = values.clip(upper=cap).astype('float64')
    return df


def confidence_bucket(count: pd.Series) -> pd.Series:
    return pd.cut(count.fillna(0), bins=[-1, 0, 9, 49, np.inf], labels=['none', 'low', 'medium', 'high']).astype('string')


def cleanup_output_files() -> None:
    for path in [BOOKS_CURATED_PATH, INTERACTIONS_CURATED_PATH, USER_FEATURES_PATH, BOOK_INTERACTION_FEATURES_PATH, SUMMARY_PATH]:
        if path.exists():
            path.unlink()


def write_parquet_chunk(path: Path, chunk: pd.DataFrame, writer: pq.ParquetWriter | None) -> pq.ParquetWriter:
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(path, table.schema, compression='snappy')
    writer.write_table(table)
    return writer

## Section 2 — Interaction Curation (global, deduplicated)

Curation of interactions is delegated to `src.curation.interactions` (single canonical cross-category artifact + separate review text + global user features). The per-genre view below is EDA-only.

In [ ]:
# Section 2 — Interaction curation (global, deduplicated)
#
# Interaction curation now lives in the reusable module `src.curation.interactions`.
# It streams the five raw dumps once and produces a SINGLE canonical, deduplicated,
# cross-category artifact (`data/processed/interactions_curated.parquet`) plus a
# separate `review_texts.parquet` and a global `user_features_global.parquet`.
# K-core and `user_rating_bias` are GLOBAL (across all categories), and the recovered
# implicit layer (`want_to_read` / `read_no_rating`) is kept and tagged.
#
# The per-genre `interactions_view.parquet` derived below is for EDA/debug only.
# NOTE: the legacy per-book interaction aggregates this notebook used to feed into the
# books cold-start flags (Section 5) are no longer produced here; Section 5 is kept as
# legacy/illustrative and is out of scope for the global interactions rebuild.
from src.config import (
    INTERACTIONS_CURATED_GLOBAL_PATH,
    REVIEW_TEXTS_PATH,
    USER_FEATURES_GLOBAL_PATH,
)
from src.curation.interactions import build_global_interactions, build_category_views

interaction_summary = build_global_interactions(force=FORCE_REPROCESS)
build_category_views(force=FORCE_REPROCESS)

interactions_view = pd.read_parquet(cfg.processed_dir / 'interactions_view.parquet')
print('Global interaction build status:', interaction_summary.get('status'))
for key in ('raw_rows', 'deduplicated_rows', 'canonical_rows', 'review_text_rows',
            'users_total', 'users_valid', 'want_to_read_pct'):
    if key in interaction_summary:
        print(f'  {key}: {interaction_summary[key]}')
print(f'Category view rows ({CATEGORY}):', len(interactions_view))
interactions_view.head()

## Section 5 — Books Pipeline and Curated Parquet

Apply EDA-grounded cleaning to all book metadata in chunks, then merge book interaction aggregates so cold-start is calculated across the full book universe.

In [7]:
def process_books_chunk(raw_books: pd.DataFrame, seen_book_ids: set[str]) -> tuple[pd.DataFrame, int]:
    books = empty_strings_to_na(raw_books.copy())
    safe_to_numeric_inplace(books, BOOK_NUMERIC_COLUMNS)

    books['book_id'] = books['book_id'].astype('string')
    if 'work_id' in books.columns:
        books['work_id'] = books['work_id'].astype('string')
    duplicate_in_chunk = int(books['book_id'].duplicated().sum())
    books = books.drop_duplicates(subset=['book_id'], keep='first')
    already_seen = books['book_id'].isin(seen_book_ids)
    duplicate_across_chunks = int(already_seen.sum())
    books = books.loc[~already_seen].copy()
    seen_book_ids.update(books['book_id'].dropna().astype(str).tolist())

    if 'is_ebook' in books.columns:
        books['is_ebook'] = parse_bool_column(books['is_ebook'])

    for col in ['title', 'title_without_series', 'description', 'publisher', 'format', 'language_code', 'country_code', 'isbn', 'isbn13', 'asin', 'kindle_asin']:
        if col in books.columns:
            books[col] = clean_string_column(books[col])

    books['language_code_clean'] = books.get('language_code', pd.Series(pd.NA, index=books.index)).astype('string').str.lower().replace('', pd.NA)
    books['format_clean'] = books.get('format', pd.Series(pd.NA, index=books.index)).astype('string').str.lower().str.strip().replace('', pd.NA)
    books['publisher_clean'] = books.get('publisher', pd.Series(pd.NA, index=books.index)).astype('string').str.lower().str.strip().replace('', pd.NA)

    for id_col in ['asin', 'kindle_asin']:
        books[f'has_{id_col}'] = books[id_col].notna() if id_col in books.columns else False

    author_features = features_from_nested(books['authors'], extract_author_features) if 'authors' in books.columns else pd.DataFrame(index=books.index)
    series_features = features_from_nested(books['series'], extract_series_features) if 'series' in books.columns else pd.DataFrame(index=books.index)
    shelf_features = features_from_nested(books['popular_shelves'], extract_shelf_features) if 'popular_shelves' in books.columns else pd.DataFrame(index=books.index)
    similar_features = features_from_nested(books['similar_books'], extract_similar_books_features) if 'similar_books' in books.columns else pd.DataFrame(index=books.index)
    books = pd.concat([books, author_features, series_features, shelf_features, similar_features], axis=1)

    pub_year = pd.to_numeric(books.get('publication_year', pd.Series(np.nan, index=books.index)), errors='coerce')
    books['publication_year_clean'] = pub_year.where(pub_year.between(PUBLICATION_YEAR_MIN, PUBLICATION_YEAR_MAX))
    books['publication_period'] = pd.cut(
        books['publication_year_clean'],
        bins=PUBLICATION_PERIOD_BINS,
        labels=PUBLICATION_PERIOD_LABELS,
        right=False,
    ).astype('string')

    theme_cols = [f'theme_{theme}' for theme in THEME_GROUPS]
    for col in theme_cols:
        books[col] = books[col].fillna(False).astype(bool)
    books[THEME_COUNT_COL] = books[THEME_COUNT_COL].fillna(0).astype('int16')
    books['series_count'] = books['series_count'].fillna(0).astype('int16')
    books['is_in_series'] = books['is_in_series'].fillna(False).astype(bool)

    books = books.drop(columns=[col for col in ['series', 'similar_books'] if col in books.columns])
    books = add_log_and_cap_features(books, ['ratings_count', 'text_reviews_count', 'to_read_count', 'num_pages'])

    books = books.merge(book_interaction_raw, on='book_id', how='left')
    for col in ['interaction_count', 'explicit_rating_count', 'review_count', 'shelf_only_count', 'read_count']:
        books[col] = books[col].fillna(0).astype('int64')
    books['mean_user_rating'] = pd.to_numeric(books['mean_user_rating'], errors='coerce')
    books['is_cold_start_all_books'] = books['interaction_count'].lt(COLD_START_THRESHOLD)
    books['is_cold_start_interacted_books'] = books['interaction_count'].gt(0) & books['interaction_count'].lt(COLD_START_THRESHOLD)
    books['interaction_confidence_bucket'] = confidence_bucket(books['interaction_count'])
    books = add_log_and_cap_features(books, ['interaction_count'])
    return books, duplicate_in_chunk + duplicate_across_chunks

In [8]:
# ── RECOVERY CHECKPOINT ──────────────────────────────────────────────────────
# Ejecutar SOLO si el kernel se desconectó después de que interactions_curated.parquet
# ya fue escrito (Section 4 completada). Reconstruye todas las variables que necesita
# el loop de books sin re-correr los passes de interacciones.
# Flujo normal: saltar esta celda.
# ─────────────────────────────────────────────────────────────────────────────

assert INTERACTIONS_CURATED_PATH.exists(), (
    f"interactions_curated.parquet no encontrado en {INTERACTIONS_CURATED_PATH}. "
    "No se puede recuperar — re-ejecutar desde Section 2."
)

_cols = [
    'user_id', 'book_id', 'rating_clean', 'engagement_mode',
    'has_review_text', 'is_read',
    'user_mean_rating', 'user_rating_std', 'user_rating_count', 'user_rating_bias',
]
_inter = pd.read_parquet(INTERACTIONS_CURATED_PATH, columns=_cols)
_inter['book_id'] = _inter['book_id'].astype('string')
_inter['user_id'] = _inter['user_id'].astype('string')

# --- user_features ---
user_features = (
    _inter[['user_id', 'user_mean_rating', 'user_rating_std', 'user_rating_count', 'user_rating_bias']]
    .drop_duplicates(subset=['user_id'])
    .reset_index(drop=True)
)
# usuarios sin ratings explícitos tienen NaN en bias — rellenar con 0
user_features['user_rating_bias'] = user_features['user_rating_bias'].fillna(0.0)
user_features['user_rating_count'] = user_features['user_rating_count'].fillna(0).astype('int64')

# --- book_interaction_raw ---
_rated = _inter[_inter['rating_clean'].notna()]
book_interaction_raw = _inter.groupby('book_id').agg(
    interaction_count=('book_id', 'size'),
    explicit_rating_count=('rating_clean', 'count'),
    review_count=('has_review_text', 'sum'),
    shelf_only_count=('engagement_mode', lambda s: (s == 'shelf_only').sum()),
    read_count=('is_read', lambda s: s.astype(bool).sum()),
).reset_index()
book_interaction_raw = book_interaction_raw.merge(
    _rated.groupby('book_id')['rating_clean'].mean().rename('mean_user_rating').reset_index(),
    on='book_id', how='left',
)
for _col in ['interaction_count', 'explicit_rating_count', 'review_count', 'shelf_only_count', 'read_count']:
    book_interaction_raw[_col] = book_interaction_raw[_col].fillna(0).astype('int64')

# --- estadísticas globales para el summary ---
second_pass_rows         = len(_inter)
second_pass_mode_counter = Counter(_inter['engagement_mode'].value_counts().to_dict())
rating_count_total       = int(_inter['rating_clean'].notna().sum())
rating_sum_total         = float(_inter['rating_clean'].sum(skipna=True))
global_mean_rating       = rating_sum_total / rating_count_total if rating_count_total else np.nan
duplicate_review_ids     = 0   # ya validado cuando se escribió interactions_curated
book_duplicate_count     = 0   # se acumulará en el loop de books

del _inter, _rated, _cols
gc.collect()

print('Recovery completado.')
print(f'  user_features:        {user_features.shape}')
print(f'  book_interaction_raw: {book_interaction_raw.shape}')
print(f'  second_pass_rows:     {second_pass_rows:,}')
print(f'  global_mean_rating:   {global_mean_rating:.4f}')
print(f'  engagement modes:     {dict(second_pass_mode_counter)}')

Recovery completado.
  user_features:        (183329, 5)
  book_interaction_raw: (23644, 7)
  second_pass_rows:     7,525,320
  global_mean_rating:   3.8604
  engagement modes:     {'rating_only': 6526843, 'review': 998477}


In [9]:
book_writer = None
book_rows = 0
book_duplicate_count = 0
seen_book_ids = set()
book_interaction_feature_parts = []
critical_null_counts = Counter()
cold_start_true_count = 0
cold_start_interacted_true_count = 0
interacted_book_count = 0

for chunk_idx, raw_books in enumerate(read_parquet_chunks(BOOKS_IN_PATH, chunksize=BOOK_CHUNKSIZE), start=1):
    books_chunk, duplicate_count = process_books_chunk(raw_books, seen_book_ids)
    book_duplicate_count += duplicate_count
    book_rows += len(books_chunk)
    cold_start_true_count += int(books_chunk['is_cold_start_all_books'].sum())
    interacted_mask = books_chunk['interaction_count'].gt(0)
    interacted_book_count += int(interacted_mask.sum())
    cold_start_interacted_true_count += int(books_chunk.loc[interacted_mask, 'is_cold_start_interacted_books'].sum())

    for col in ['book_id', 'primary_author_id_role_filtered', 'publication_year_clean', 'to_read_count']:
        critical_null_counts[f'books.{{col}}'] += int(books_chunk[col].isna().sum())

    book_interaction_feature_parts.append(books_chunk[[
        'book_id',
        'interaction_count',
        'explicit_rating_count',
        'review_count',
        'shelf_only_count',
        'read_count',
        'mean_user_rating',
        'is_cold_start_all_books',
        'is_cold_start_interacted_books',
        'interaction_confidence_bucket',
    ]].copy())

    book_writer = write_parquet_chunk(BOOKS_CURATED_PATH, books_chunk, book_writer)

    if chunk_idx % 5 == 0:
        print(f'Book chunks={{chunk_idx:,}} books={{book_rows:,}}')

    del raw_books, books_chunk
    gc.collect()

if book_writer is not None:
    book_writer.close()

book_interaction_features = pd.concat(book_interaction_feature_parts, ignore_index=True)
safe_write_parquet(book_interaction_features, BOOK_INTERACTION_FEATURES_PATH)
safe_write_parquet(user_features, USER_FEATURES_PATH)

print(f'Wrote books: {{book_rows:,}} rows -> {{BOOKS_CURATED_PATH}}')
print(f'Book duplicate count observed: {{book_duplicate_count:,}}')

Wrote books: {book_rows:,} rows -> {BOOKS_CURATED_PATH}
Book duplicate count observed: {book_duplicate_count:,}


## Section 6 — Summary

Write a JSON summary for reproducibility and quick downstream inspection.

In [ ]:
curation_summary = {
    'category': CATEGORY,
    'display_name': cfg.display_name,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'books_rows': int(book_rows),
    'interactions_global_canonical_rows': int(interaction_summary.get('canonical_rows', 0)),
    'interactions_global_users_valid': int(interaction_summary.get('users_valid', 0)),
    'global_mean_rating': interaction_summary.get('global_mean_rating'),
    'want_to_read_pct': interaction_summary.get('want_to_read_pct'),
    'engagement_mode_counts': interaction_summary.get('engagement_mode_counts', {}),
    'critical_null_counts': dict(critical_null_counts),
    'cold_start_all_books_pct': cold_start_true_count / book_rows if book_rows else 0.0,
    'cold_start_interacted_books_pct': cold_start_interacted_true_count / interacted_book_count if interacted_book_count else 0.0,
    'duplicate_book_ids_observed': int(book_duplicate_count),
    'output_files': {
        'books_curated': str(BOOKS_CURATED_PATH),
        'interactions_curated_global': str(INTERACTIONS_CURATED_GLOBAL_PATH),
        'review_texts': str(REVIEW_TEXTS_PATH),
        'user_features_global': str(USER_FEATURES_GLOBAL_PATH),
    },
}
SUMMARY_PATH.write_text(json.dumps(curation_summary, ensure_ascii=False, indent=2))
print(json.dumps(curation_summary, ensure_ascii=False, indent=2)[:2000])

## Section 7 — Validation

Assertions mirror the EDA-grounded test plan and protect the downstream recommender from silent data-contract drift. Large parquet files are validated by reading only the needed columns.

In [ ]:
# --- Interaction validation: the global, deduplicated canonical artifact ---
from src.config import (
    INTERACTIONS_CURATED_GLOBAL_PATH,
    REVIEW_TEXTS_PATH,
    USER_FEATURES_GLOBAL_PATH,
)

CANONICAL_ENGAGEMENT_MODES = {'want_to_read', 'read_no_rating', 'rating_only', 'review'}

assert BOOKS_CURATED_PATH.exists()
assert INTERACTIONS_CURATED_GLOBAL_PATH.exists()
assert REVIEW_TEXTS_PATH.exists()
assert USER_FEATURES_GLOBAL_PATH.exists()

canonical = pd.read_parquet(
    INTERACTIONS_CURATED_GLOBAL_PATH,
    columns=['interaction_key', 'rating_clean', 'rating_missing', 'engagement_mode',
             'is_want_to_read', 'user_rating_bias', 'has_review_text'],
)
assert 'review_text_clean' not in canonical.columns  # text lives in review_texts.parquet
assert canonical['interaction_key'].is_unique
assert canonical['rating_clean'].dropna().between(1, 5).all()
assert set(canonical['engagement_mode'].unique()).issubset(CANONICAL_ENGAGEMENT_MODES)
# implicit layer recovered (legacy curated had 0% rating_missing / 100% is_read)
assert canonical['rating_missing'].any()
assert {'want_to_read', 'read_no_rating'} & set(canonical['engagement_mode'].unique())
assert canonical['user_rating_bias'].notna().all()

user_features_global = pd.read_parquet(USER_FEATURES_GLOBAL_PATH, columns=['user_rating_bias', 'valid'])
assert user_features_global['user_rating_bias'].notna().all()

review_texts = pd.read_parquet(REVIEW_TEXTS_PATH, columns=['interaction_key', 'review_text_length'])
assert review_texts['interaction_key'].is_unique
assert (review_texts['review_text_length'] > 0).all()

books_check_cols = ['book_id', 'authors', 'popular_shelves', 'primary_author_id_role_filtered', 'author_count', 'publication_year_clean', 'to_read_count', THEME_COUNT_COL, 'interaction_count', 'is_cold_start_all_books', 'is_cold_start_interacted_books'] + theme_cols
books_check = pd.read_parquet(BOOKS_CURATED_PATH, columns=books_check_cols)
assert books_check['book_id'].notna().all()
assert books_check['book_id'].duplicated().sum() == 0
assert books_check['publication_year_clean'].dropna().between(PUBLICATION_YEAR_MIN, PUBLICATION_YEAR_MAX).all()
for col in theme_cols:
    assert books_check[col].dtype == bool
assert THEME_COUNT_COL in books_check.columns
assert books_check.loc[books_check['interaction_count'].eq(0), 'is_cold_start_all_books'].all()
assert books_check['is_cold_start_all_books'].dtype == bool
assert books_check['is_cold_start_interacted_books'].dtype == bool

print('All validations passed!')